# Open-Weight LLM Deployment Pipeline — Kaggle E2E

Run this notebook in Kaggle with Internet enabled. A GPU accelerator is recommended but not mandatory for the API/unit-test stages. The default model is HuggingFaceTB/SmolLM2-360M. Model files are cached only in the Kaggle session and are not committed to Git.

In [ ]:
import os, sys, json, time, platform, subprocess, pathlib, statistics
from datetime import datetime, timezone

REPO_URL = "https://github.com/razz1992/OpenWeight-LLM-Deployment-Pipeline.git"
REPO_DIR = pathlib.Path("/kaggle/working/OpenWeight-LLM-Deployment-Pipeline")
MODEL_ID = "HuggingFaceTB/SmolLM2-360M"
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Started UTC:", datetime.now(timezone.utc).isoformat())


In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print("Repository:", REPO_DIR)
print(subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())


In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("Running CPU path; model inference may be slower.")


## 1. Install project dependencies

Kaggle images often include PyTorch already. This installs the project API/test/model libraries without reinstalling PyTorch.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Dependencies installed.")


## 2. API unit and validation tests

These tests do not require model weights or a GPU. They verify health response, unloaded-model handling and request validation.

In [ ]:
test_result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/unit"], capture_output=True, text=True)
print(test_result.stdout)
print(test_result.stderr)
assert test_result.returncode == 0, "Unit tests failed"


## 3. Model metadata and license review

Review the model card and license before using or redistributing weights. This cell retrieves the repository metadata; it does not grant redistribution rights.

In [ ]:
from huggingface_hub import HfApi
info = HfApi().model_info(MODEL_ID)
model_metadata = {"model_id": MODEL_ID, "revision": info.sha, "pipeline_tag": info.pipeline_tag,
                  "license": (info.card_data.license if info.card_data else None),
                  "private": info.private}
print(json.dumps(model_metadata, indent=2))
assert not info.private, "Expected a public model repository"


## 4. Load model and run inference

Use float16 on CUDA and float32 on CPU. The model is intentionally small to fit typical free Kaggle sessions.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch_dtype, device_map="auto" if torch.cuda.is_available() else None)
if not torch.cuda.is_available():
    model.to("cpu")
model.eval()
load_seconds = time.perf_counter() - t0
print("Loaded in seconds:", round(load_seconds, 3))


In [ ]:
def generate_text(prompt, max_new_tokens=96, temperature=0.0):
    inputs = tokenizer(prompt, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    kwargs = {"max_new_tokens": max_new_tokens, "do_sample": temperature > 0, "pad_token_id": tokenizer.eos_token_id}
    if temperature > 0:
        kwargs["temperature"] = temperature
    with torch.inference_mode():
        output = model.generate(**inputs, **kwargs)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

prompt = "Explain why monitoring latency matters when serving a language model:"
t0 = time.perf_counter()
answer = generate_text(prompt)
first_inference_seconds = time.perf_counter() - t0
print("Prompt:", prompt)
print("Output:", answer)
print("First inference seconds:", round(first_inference_seconds, 3))
assert isinstance(answer, str) and len(answer.strip()) > 0


## 5. Benchmark and resource telemetry

This is a small smoke benchmark, not a statistically rigorous performance study. Repeat with more prompts and warm-up iterations for production comparisons.

In [ ]:
prompts = ["What is an open-weight language model?", "Explain model quantization in simple terms.", "What does an inference API do?"]
latencies = []
for p in prompts:
    t0 = time.perf_counter(); out = generate_text(p, max_new_tokens=64); latencies.append(time.perf_counter()-t0)
    print({"prompt": p, "latency_seconds": round(latencies[-1], 3), "output_chars": len(out)})
try:
    import psutil
    ram_gib = psutil.virtual_memory().used / 1024**3
except Exception:
    ram_gib = None
if torch.cuda.is_available():
    gpu_peak_gib = torch.cuda.max_memory_allocated() / 1024**3
else:
    gpu_peak_gib = None
benchmark = {"load_seconds": load_seconds, "first_inference_seconds": first_inference_seconds,
             "latency_seconds": latencies, "mean_latency_seconds": statistics.mean(latencies),
             "system_ram_used_gib": ram_gib, "gpu_peak_allocated_gib": gpu_peak_gib}
print(json.dumps(benchmark, indent=2))


## 6. Exercise the FastAPI endpoint with the loaded model

The API unit tests above validated the unloaded state. Here we inject the already-loaded notebook model into the API service and test a real generation request using FastAPI TestClient.

In [ ]:
from fastapi.testclient import TestClient
from api.main import app, model_service

# Adapt the notebook model to the service's generator interface.
def notebook_generator(prompt, **kwargs):
    generated = generate_text(prompt, max_new_tokens=kwargs.get("max_new_tokens", 96), temperature=kwargs.get("temperature", 0.0))
    return [{"generated_text": prompt + generated}]

model_service.generator = notebook_generator
client = TestClient(app)
health = client.get("/health")
api_response = client.post("/generate", json={"prompt": "Define inference in one sentence:", "max_new_tokens": 48, "temperature": 0.0})
print("Health:", health.status_code, health.json())
print("Generate status:", api_response.status_code)
print("Generate body:", api_response.json())
assert health.status_code == 200 and health.json()["model_loaded"] is True
assert api_response.status_code == 200
assert api_response.json()["generated_text"].strip()


## 7. LLM evaluation — fixed golden dataset

Evaluate response quality, instruction adherence, hallucination awareness and performance using eight repeatable prompts. Quantization is intentionally skipped.

In [ ]:
import time
import json
import statistics
from datetime import datetime, timezone

evaluation_cases = [
    {"id": "E01", "category": "Technical knowledge", "prompt": "Explain the difference between data drift and model drift in 3 bullet points."},
    {"id": "E02", "category": "Technical explanation", "prompt": "Explain how an LLM inference API works in simple terms."},
    {"id": "E03", "category": "Instruction following", "prompt": "Explain RAG in exactly 3 numbered steps. Keep each step under 15 words."},
    {"id": "E04", "category": "Summarization", "prompt": "Summarize in one sentence: An application receives user prompts, sends them to a language model, receives generated responses, and returns those responses to users."},
    {"id": "E05", "category": "Reasoning", "prompt": "An API receives 120 requests in 10 seconds. What is its average requests per second? Explain the calculation."},
    {"id": "E06", "category": "Hallucination awareness", "prompt": "What was the exact GPU temperature of my Kaggle session yesterday? If you do not have this information, say so."},
    {"id": "E07", "category": "Code understanding", "prompt": "Explain what this Python code does: numbers = [1, 2, 3]; result = [x * 2 for x in numbers]"},
    {"id": "E08", "category": "Conciseness", "prompt": "Define LLMOps in no more than 30 words."}
]

evaluation_results = []
_ = generate_text("Explain what an API is:", max_new_tokens=32)  # warm-up excluded

for case in evaluation_cases:
    start = time.perf_counter()
    response = generate_text(case["prompt"], max_new_tokens=128, temperature=0.0)
    latency = time.perf_counter() - start
    output_tokens = len(tokenizer.encode(response, add_special_tokens=False))
    evaluation_results.append({
        "id": case["id"], "category": case["category"], "prompt": case["prompt"],
        "response": response, "latency_seconds": round(latency, 3),
        "output_tokens": output_tokens,
        "tokens_per_second": round(output_tokens / latency, 2) if latency > 0 else 0
    })
    print("\n" + "=" * 70)
    print(case["id"], "-", case["category"])
    print("Prompt:", case["prompt"])
    print("Response:", response)
    print("Latency:", round(latency, 3), "seconds")
    print("Output tokens:", output_tokens)
    print("Tokens/second:", round(output_tokens / latency, 2) if latency > 0 else 0)

print("\nGenerated", len(evaluation_results), "evaluation responses.")

## 8. Manual quality scoring

Review the eight responses above. Score each dimension from 1 (poor) to 5 (excellent). For E06, reward the model for acknowledging that it cannot know yesterday's GPU temperature from this prompt.

In [ ]:
# Enter integer scores from 1 to 5 after reviewing each response.
# correctness: factual accuracy; relevance: answers the question;
# instruction_adherence: follows format/constraints; completeness: covers key points.
manual_scores = {
    "E01": {"correctness": None, "relevance": None, "instruction_adherence": None, "completeness": None},
    "E02": {"correctness": None, "relevance": None, "instruction_adherence": None, "completeness": None},
    "E03": {"correctness": None, "relevance": None, "instruction_adherence": None, "completeness": None},
    "E04": {"correctness": None, "relevance": None, "instruction_adherence": None, "completeness": None},
    "E05": {"correctness": None, "relevance": None, "instruction_adherence": None, "completeness": None},
    "E06": {"correctness": None, "relevance": None, "instruction_adherence": None, "completeness": None},
    "E07": {"correctness": None, "relevance": None, "instruction_adherence": None, "completeness": None},
    "E08": {"correctness": None, "relevance": None, "instruction_adherence": None, "completeness": None}
}
print("Review responses, enter all scores from 1 to 5, then run the report cell.")

## 9. Generate and save the LLM evaluation report

Run this cell only after entering all manual scores above.

In [ ]:
import pandas as pd
score_rows = []
for result in evaluation_results:
    scores = manual_scores[result["id"]]
    if any(not isinstance(v, int) or isinstance(v, bool) or not 1 <= v <= 5 for v in scores.values()):
        raise ValueError(f"Complete all four scores for {result['id']} using integers 1 to 5.")
    score_rows.append({
        "id": result["id"], "category": result["category"], **scores,
        "quality_score": round(statistics.mean(scores.values()), 2),
        "latency_seconds": result["latency_seconds"],
        "output_tokens": result["output_tokens"],
        "tokens_per_second": result["tokens_per_second"]
    })

evaluation_df = pd.DataFrame(score_rows)
display(evaluation_df)
summary = {
    "project": "Open-Weight LLM Deployment Pipeline", "model": MODEL_ID,
    "evaluation_date_utc": datetime.now(timezone.utc).isoformat(),
    "total_test_cases": len(evaluation_df),
    "average_quality_score": round(evaluation_df["quality_score"].mean(), 2),
    "average_latency_seconds": round(evaluation_df["latency_seconds"].mean(), 3),
    "average_tokens_per_second": round(evaluation_df["tokens_per_second"].mean(), 2),
    "scoring_method": "Manual human scoring; four dimensions, each 1-5",
    "evaluation_results": score_rows, "responses": evaluation_results
}
json_path = pathlib.Path("/kaggle/working/llm_evaluation_report.json")
csv_path = pathlib.Path("/kaggle/working/llm_evaluation_scores.csv")
json_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
evaluation_df.to_csv(csv_path, index=False)
print("Saved:", json_path)
print("Saved:", csv_path)
print("Average quality score:", summary["average_quality_score"], "/ 5")

## 10. Evaluation notes

- This is a small baseline evaluation, not a statistically representative benchmark.
- Temperature is fixed at 0 for repeatability.
- Manual scoring is used rather than an LLM judge.
- Token throughput is generated output tokens divided by wall-clock generation time.
- Compare models only with the same prompts, generation limits, hardware and scoring rubric.